In [18]:
import torch
import numpy as np
import pandas as pd
from haversine import haversine, Unit
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool
import torch.nn.functional as F
from sklearn.preprocessing import LabelEncoder, StandardScaler
from graphrfi_subgraphs import *


In [19]:
partition = 100

# 1. Load Dataset

In [20]:
trainpath = f'../../../data/top30groups/LongLatCombined/train1/train{partition}.csv'
testpath = f'../../../data/top30groups/LongLatCombined/test1/test{partition}.csv'
traindata = pd.read_csv(trainpath, encoding='ISO-8859-1')
testdata = pd.read_csv(testpath, encoding='ISO-8859-1')

In [21]:
traindata_list, testdata_list, y_gcn, y_nrf, nrf_input, index_to_label = build_graph_data(traindata, testdata, 'weaptype1')

Feature Matrix shape:  (1790, 2)


In [22]:
from itertools import product

param_grid = {
    'embed_dim': [16,32],
    'lr': [0.01],
    'n_tree': [80],
    'tree_depth': [12],
    'feat_dropout': [0.1],
    'tree_feature_rate': [0.2],
    'batch_size': [256]
}

In [23]:
import copy

best_acc = -1
best_params = None
best_epoch = -1
results = []

# Create all combinations of the parameter grid
keys, values = zip(*param_grid.items())
i = 1
total_combinations = len(list(product(*values)))

for v in product(*values):
    # Build argument dict
    print(f"{i}/{total_combinations}")
    params = dict(zip(keys, v))
    
    # Merge with fixed defaults
    args = {
        'partition': f"gtd{partition}",
        'epochs': 3000
        ,
        'n_class': 30,
        'final_evaluation': True,
        **params  # override with params from grid
    }

    print(f"\nRunning: {args}")

    try:
        acc, epoch, *_ = train_joint_subgraph(
            traindata_list,
            testdata_list,
            y_gcn,
            y_nrf,
            nrf_input,
            args,
            index_to_label,
            verbose=False
        )

        results.append((acc, copy.deepcopy(args)))

        if acc > best_acc:
            best_acc = acc
            best_epoch = epoch
            best_params = copy.deepcopy(args)

    except Exception as e:
        print(f"Error with params {params}: {e}")

    i = i + 1

print("\nBest Accuracy:", best_acc)
print("\nBest Epoch:", best_epoch)
print("Best Parameters:")
for k, v in best_params.items():
    print(f"{k}: {v}")


1/2

Running: {'partition': 'gtd100', 'epochs': 3000, 'n_class': 30, 'final_evaluation': True, 'embed_dim': 16, 'lr': 0.01, 'n_tree': 80, 'tree_depth': 12, 'feat_dropout': 0.1, 'tree_feature_rate': 0.2, 'batch_size': 256}
Early stopping at epoch 2226
Best acc/epoch: 0.7500 at epoch 1726
Predicting on test set
2/2

Running: {'partition': 'gtd100', 'epochs': 3000, 'n_class': 30, 'final_evaluation': True, 'embed_dim': 32, 'lr': 0.01, 'n_tree': 80, 'tree_depth': 12, 'feat_dropout': 0.1, 'tree_feature_rate': 0.2, 'batch_size': 256}
Early stopping at epoch 2077
Best acc/epoch: 0.7357 at epoch 1577
Predicting on test set

Best Accuracy: 0.7088888883590698

Best Epoch: 1577
Best Parameters:
partition: gtd100
epochs: 3000
n_class: 30
final_evaluation: True
embed_dim: 32
lr: 0.01
n_tree: 80
tree_depth: 12
feat_dropout: 0.1
tree_feature_rate: 0.2
batch_size: 256


In [24]:
"""Running: {'partition': 'gtd100', 'epochs': 3000, 'n_class': 30, 'final_evaluation': True, 'embed_dim': 16, 'lr': 0.001, 'n_tree': 80, 'tree_depth': 10, 'feat_dropout': 0.1, 'tree_feature_rate': 0.5, 'batch_size': 256}
Early stopping at epoch 1815
Best acc/epoch: 0.7976 at epoch 1315
Predicting on test set

Best Accuracy: 0.7822222709655762"""

"Running: {'partition': 'gtd100', 'epochs': 3000, 'n_class': 30, 'final_evaluation': True, 'embed_dim': 16, 'lr': 0.001, 'n_tree': 80, 'tree_depth': 10, 'feat_dropout': 0.1, 'tree_feature_rate': 0.5, 'batch_size': 256}\nEarly stopping at epoch 1815\nBest acc/epoch: 0.7976 at epoch 1315\nPredicting on test set\n\nBest Accuracy: 0.7822222709655762"

In [25]:
"""partition: gtd100
epochs: 1500
n_class: 30
embed_dim: 32
lr: 0.001
n_tree: 80
tree_depth: 10
feat_dropout: 0
tree_feature_rate: 0.3
batch_size: 256
0.8355555534362793"""

args = {
    'partition': f"gtd{partition}",
    'epochs': 3000,
    'n_class': 30,
    'final_evaluation': True,
    'lr': 0.001,
    'embed_dim': 32,
    'n_tree': 80,
    'tree_depth': 10,
    'feat_dropout': 0,
    'tree_feature_rate': 0.3,
    'batch_size': 256
}

best_acc,best_epoch,precision, recall, f1,y_pred_decoded, y_true_decoded,precision_micro, recall_micro, f1_micro,precision_macro, recall_macro, f1_macro,roc_auc_weighted, roc_auc_micro, roc_auc_macro,epoch_logs = train_joint_subgraph(
        traindata_list,
        testdata_list,
        y_gcn,
        y_nrf,
        nrf_input,
        args,
        index_to_label,
        verbose=True
    )

print(best_acc, best_epoch)

metrics = {
    "best_acc": [best_acc],
    "best_epoch": [best_epoch],
    "precision_weighted": [precision],
    "recall_weighted": [recall],
    "f1_weighted": [f1],
    "precision_micro": [precision_micro],
    "recall_micro": [recall_micro],
    "f1_micro": [f1_micro],
    "precision_macro": [precision_macro],
    "recall_macro": [recall_macro],
    "f1_macro": [f1_macro],
    "roc_auc_weighted": [roc_auc_weighted],
    "roc_auc_micro": [roc_auc_micro],
    "roc_auc_macro": [roc_auc_macro],
}

pd.DataFrame(metrics).to_csv("results.csv", index=False)

Epoch 01 | Joint Loss: 39.9900 | NRF Acc: 0.1571
Epoch 02 | Joint Loss: 38.3873 | NRF Acc: 0.1357
Epoch 51 | Joint Loss: 4.2251 | NRF Acc: 0.3905
Epoch 101 | Joint Loss: 3.7554 | NRF Acc: 0.4690
Epoch 151 | Joint Loss: 3.2458 | NRF Acc: 0.6071
Epoch 201 | Joint Loss: 3.0313 | NRF Acc: 0.6190
Epoch 251 | Joint Loss: 2.8515 | NRF Acc: 0.6333
Epoch 301 | Joint Loss: 2.7206 | NRF Acc: 0.6548
Epoch 351 | Joint Loss: 2.6115 | NRF Acc: 0.6595
Epoch 401 | Joint Loss: 2.5532 | NRF Acc: 0.6452
Epoch 451 | Joint Loss: 2.4704 | NRF Acc: 0.6643
Epoch 501 | Joint Loss: 2.4461 | NRF Acc: 0.6881
Epoch 551 | Joint Loss: 2.3666 | NRF Acc: 0.7143
Epoch 601 | Joint Loss: 2.3268 | NRF Acc: 0.6810
Epoch 651 | Joint Loss: 2.2816 | NRF Acc: 0.6714
Epoch 701 | Joint Loss: 2.2299 | NRF Acc: 0.7167
Epoch 751 | Joint Loss: 2.2009 | NRF Acc: 0.7167
Epoch 801 | Joint Loss: 2.1550 | NRF Acc: 0.7214
Epoch 851 | Joint Loss: 2.1706 | NRF Acc: 0.7357
Epoch 901 | Joint Loss: 2.1122 | NRF Acc: 0.7238
Epoch 951 | Joint Los

KeyboardInterrupt: 

In [ ]:
#best_acc, best_epoch, precision, recall, f1, y_pred_decoded, y_true_decoded, precision_micro, recall_micro, f1_micro,precision_macro, recall_macro, f1_macro,roc_auc_weighted, roc_auc_micro, roc_auc_macro,epoch_logs = train_joint_subgraph(traindata_list, testdata_list, y_gcn, y_nrf, nrf_input, default_args, index_to_label, verbose=True)

In [ ]:
"""
default_args = {
    'partition': f"gtd{partition}",
    'embed_dim': 16,
    'lr': 0.001,
    'epochs': 1500,
    'feat_dropout': 0,
    'n_tree': 80,
    'tree_depth': 10,
    'tree_feature_rate': 0.5,
    'n_class': 30,
    'batch_size': 256
}
"""
#Best acc/epoch: 0.8333 at epoch 1454


'\ndefault_args = {\n    \'partition\': f"gtd{partition}",\n    \'embed_dim\': 16,\n    \'lr\': 0.001,\n    \'epochs\': 1500,\n    \'feat_dropout\': 0,\n    \'n_tree\': 80,\n    \'tree_depth\': 10,\n    \'tree_feature_rate\': 0.5,\n    \'n_class\': 30,\n    \'batch_size\': 256\n}\n'

In [ ]:
#Best acc/epoch: 0.7856 at epoch 400
#Best acc/epoch: 0.8133 at epoch 1378


In [ ]:
best_acc

0.7744444608688354